# Clean text and create word tuples for later BPE

In [27]:
import re


save_path = "wiki.train.tokens"

words_set = []
vocabulary = set()
right_symbols = re.compile(r"[a-zA-Z,()?!\d]")

with open(save_path, "r") as f:
    for i, row in enumerate(f):
        if i % 1000 == 0:
            print(i)

        if row.startswith(" =") or row.startswith("="):
            continue
        
        token = []
        row = re.sub(r"<[^>]+>", "", row)
        row = re.sub(r"[^a-zA-Z0-9 .,!?]", "", row)
        for sym_id, symbol in enumerate(row):
            vocabulary.add(symbol)
            token = token + [symbol]
            if symbol == " " or sym_id == len(row) - 1:
                if token not in words_set:
                    words_set.append(token)
                token = []

print(len(vocabulary))
len(words_set)

0
1000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
18000
19000
21000
22000
23000
24000
25000
26000
28000
29000
30000
31000
32000
34000
35000
36000
38000
40000
41000
42000
43000
44000
45000
47000
48000
49000
50000
51000
52000
53000
55000
56000
57000
58000
59000
60000
61000
62000
64000
66000
67000
68000
69000
70000
71000
72000
73000
75000
76000
78000
79000
80000
82000
84000
85000
88000
89000
90000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
116000
117000
118000
120000
121000
122000
123000
124000
125000
127000
128000
129000
130000
131000
132000
134000
135000
137000
138000
139000
140000
141000
143000
144000
145000
146000
147000
149000
150000
151000
152000
153000
154000
155000
156000
157000
160000
161000
162000
163000
164000
165000
166000
167000
168000
169000
171000
172000
173000
174000
175000
176000
177000
178000
179000
180000
181000
182000
183000
184000
185000
1

263525

In [ ]:
# Cleaned version save
import json


with open("cleaned_text.json", "w") as f:
    json.dump(words_set, f)

In [38]:
with open("cleaned_text.json", "r") as f:
    words_set = json.load(f)

words_set[10:30]

vocabulary = set()
for word in words_set:
    for symbol in word:
        vocabulary.add(symbol)

print(len(vocabulary))
print(vocabulary)

67
{'s', 'C', '!', 'F', 'm', 'Q', 'u', 'J', ',', 'y', 'a', 'r', '5', 'p', 'W', 'i', 'n', 'K', 'N', '8', 'w', 'g', '.', 'R', 'l', 'B', 'd', 'e', 'j', 'o', 'P', '9', 't', 'h', '3', 'O', '2', '?', 'Z', 'G', 'V', 'U', 'M', 'D', '0', 'X', '6', 'A', ' ', 'H', 'L', 'v', 'T', '7', 'b', 'Y', 'I', 'c', 'E', 'S', 'f', 'k', 'x', 'z', '4', '1', 'q'}


# Actual BPE logic

In [39]:
last_merged = None
vocabulary_size = 36_000
while len(vocabulary) <= vocabulary_size:
    if len(vocabulary) % 200 == 0:
        print(len(vocabulary))

    frequency = {}

    for w_id, word in enumerate(words_set):
        i = 0
        while i < len(word) - 1:

            symbol_pair = word[i] + word[i + 1]

            # automatically merge
            if last_merged and symbol_pair == last_merged:
                word[i] = word[i] + word[i + 1]
                del word[i + 1]

                if i > 0:
                    i -= 1
                continue

            frequency[symbol_pair] = frequency.get(symbol_pair, 0) + 1
            i += 1

    last_merged = max(frequency, key=frequency.get)
    vocabulary.add(last_merged)

print(len(vocabulary))
words_set[20:30]

5
5
5
4
4
4
4
4
4
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
36001


[['is '],
 ['a '],
 ['tactical '],
 ['r', 'ole '],
 ['playing '],
 ['vide', 'o '],
 ['game '],
 ['developed '],
 ['by '],
 ['Seg', 'a ']]

# Create STOI and encode text

In [44]:
stoi = {token: i for i, token in enumerate(vocabulary)}

encoded_text = []
for word in words_set:
    for token in word: 
        encoded_text.append(stoi[token])

print(f"Small sample of encoded test: {encoded_text[10:15]}")
print(f"Length of the encdoed text: {len(encoded_text)}")

Small sample of encoded test: [2586, 26700, 8757, 10498, 24658]
Length of the encdoed text: 589545


# Save to file

In [45]:
import json
import numpy as np


np_encoded_text = np.array(encoded_text, dtype=np.uint32)
np_encoded_text.tofile("train.bin")

with open("stoi.json", "w") as f:
    json.dump(stoi, f)